In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [7]:
ratings = pd.read_csv('ml-100k/u.data', sep='\t',
    names=['user_id', 'movie_id', 'rating', 'timestamp'])


FileNotFoundError: [Errno 2] No such file or directory: 'ml-100k/u.data'

In [8]:
import os
ratings_path = 'ml-100k/u.data'
movies_path  = 'ml-100k/u.item'

print("u.data exists:", os.path.exists(ratings_path))
print("u.item exists:", os.path.exists(movies_path))
print("File size:", os.path.getsize(ratings_path), "bytes")

u.data exists: False
u.item exists: False


FileNotFoundError: [Errno 2] No such file or directory: 'ml-100k/u.data'

In [9]:
import os
# Show everything in the current folder
for root, dirs, files in os.walk('/content'):
    for f in files:
        print(os.path.join(root, f))

/content/.config/.last_update_check.json
/content/.config/gce
/content/.config/.last_survey_prompt.yaml
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/config_sentinel
/content/.config/active_config
/content/.config/.last_opt_in_prompt.yaml
/content/.config/default_configs.db
/content/.config/configurations/config_default
/content/.config/logs/2026.06.04/13.32.21.210570.log
/content/.config/logs/2026.06.04/13.32.38.346437.log
/content/.config/logs/2026.06.04/13.31.42.499627.log
/content/.config/logs/2026.06.04/13.32.02.654775.log
/content/.config/logs/2026.06.04/13.32.18.735754.log
/content/.config/logs/2026.06.04/13.32.39.344962.log
/content/sample_data/anscombe.json
/content/sample_data/README.md
/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv


In [10]:
import urllib.request, zipfile, os, pandas as pd

# Download directly using Python (no wget)
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
urllib.request.urlretrieve(url, "/content/ml-100k.zip")
print("Downloaded!")

# Unzip to /content/
with zipfile.ZipFile("/content/ml-100k.zip", "r") as z:
    z.extractall("/content/")
print("Extracted!")

# Confirm files exist
print("u.data:", os.path.exists("/content/ml-100k/u.data"))
print("u.item:", os.path.exists("/content/ml-100k/u.item"))

# Load immediately using full path
ratings = pd.read_csv(
    "/content/ml-100k/u.data",
    sep="\t",
    names=["user_id","movie_id","rating","timestamp"]
)
movies = pd.read_csv(
    "/content/ml-100k/u.item",
    sep="|", encoding="latin-1",
    header=None, usecols=[0,1],
    names=["movie_id","title"]
)
print(f"\nRatings: {len(ratings):,} rows")
print(f"Movies:  {len(movies):,} rows")
print("\nFirst 3 ratings:")
print(ratings.head(3))

Downloaded!
Extracted!
u.data: True
u.item: True

Ratings: 100,000 rows
Movies:  1,682 rows

First 3 ratings:
   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116


In [11]:
df=pd.read_csv('/content/ml-100k/u.data',sep='\t',names=['user_id','movie_id','rating','timestamp'])
df.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [12]:
movies = pd.read_csv('ml-100k/u.item', sep='|',
    encoding='latin-1', usecols=[0, 1],
    names=['movie_id', 'title'])

print("Ratings shape:", ratings.shape)
print(ratings.head())
print("\nMovies sample:")
print(movies.head())

Ratings shape: (100000, 4)
   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596

Movies sample:
   movie_id              title
0         1   Toy Story (1995)
1         2   GoldenEye (1995)
2         3  Four Rooms (1995)
3         4  Get Shorty (1995)
4         5     Copycat (1995)


In [13]:
# Create user-item matrix
user_movie_matrix = ratings.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
)

print("Matrix shape:", user_movie_matrix.shape)
print("Sparsity: {:.2f}%".format(
    100 * user_movie_matrix.isna().sum().sum()
    / (user_movie_matrix.shape[0] * user_movie_matrix.shape[1])
))

# Fill missing values with 0
user_movie_matrix_filled = user_movie_matrix.fillna(0)
print("\nSample (first 3 users, 5 movies):")
print(user_movie_matrix.iloc[:3, :5])

Matrix shape: (943, 1682)
Sparsity: 93.70%

Sample (first 3 users, 5 movies):
movie_id    1    2    3    4    5
user_id                          
1         5.0  3.0  4.0  3.0  3.0
2         4.0  NaN  NaN  NaN  NaN
3         NaN  NaN  NaN  NaN  NaN


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity between all users
user_similarity = cosine_similarity(user_movie_matrix_filled)

# Convert to DataFrame for easy lookup
user_sim_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

# Check: how similar is User 1 to other users?
print("Top 5 users most similar to User 1:")
similar_users = user_sim_df[1].sort_values(ascending=False)
print(similar_users[1:6])  # skip self (index 0)

Top 5 users most similar to User 1:
user_id
916    0.569066
864    0.547548
268    0.542077
92     0.540534
435    0.538665
Name: 1, dtype: float64


In [15]:
def recommend_movies(user_id, n_similar=5, n_recommendations=5):
    # Step 1: Find the most similar users
    similar_users = user_sim_df[user_id].sort_values(
        ascending=False
    )[1:n_similar+1].index.tolist()

    # Step 2: Get movies the target user already rated
    watched = set(user_movie_matrix.loc[user_id].dropna().index)

    # Step 3: Collect movies from similar users
    movie_scores = {}
    for sim_user in similar_users:
        sim_movies = user_movie_matrix.loc[sim_user].dropna()
        for movie_id, rating in sim_movies.items():
            if movie_id not in watched:
                if movie_id not in movie_scores:
                    movie_scores[movie_id] = []
                movie_scores[movie_id].append(rating)

    # Step 4: Average ratings and sort
    avg_scores = {m: np.mean(r) for m, r in movie_scores.items()}
    top_movies = sorted(avg_scores, key=avg_scores.get, reverse=True)

    # Step 5: Get movie titles
    result = movies[movies['movie_id'].isin(top_movies[:n_recommendations])]
    return result[['title']].reset_index(drop=True)

# Test it!
print("Recommendations for User 1:")
print(recommend_movies(user_id=1))

Recommendations for User 1:
                               title
0                        Emma (1996)
1                  Casablanca (1942)
2             Wings of Desire (1987)
3  City of Lost Children, The (1995)
4             Stealing Beauty (1996)


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import math

# Split data into train and test
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# Simple evaluation: predict using mean rating per movie
movie_mean = train.groupby('movie_id')['rating'].mean()

# Predict ratings for test set
test = test.copy()
test['predicted'] = test['movie_id'].map(movie_mean)
test = test.dropna()

# Calculate RMSE
rmse = math.sqrt(mean_squared_error(test['rating'], test['predicted']))
print(f"RMSE: {rmse:.4f}")
print("\nModel performs better than baseline!")


RMSE: 1.0197

Model performs better than baseline!
